In [1]:
import os
import urllib.request
from dotenv import load_dotenv
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import json
from datetime import datetime, timezone

# ==========================================
# 1. CONFIGURAÇÃO DO JAVA 17 COMPATÍVEL
# ==========================================
# Ajuste o caminho abaixo se a sua pasta real do JDK 17 possuir outro nome
os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-17.0.19.10-hotspot"
os.environ["PATH"] = os.environ["JAVA_HOME"] + r"\bin;" + os.environ["PATH"]


# ==========================================
# 2. CONFIGURAÇÃO AUTOMÁTICA DO HADOOP (Windows)
# ==========================================
hadoop_dir = r"C:\hadoop"
hadoop_bin = os.path.join(hadoop_dir, "bin")

# Cria a estrutura de pastas se não existir
os.makedirs(hadoop_bin, exist_ok=True)

# URLs oficiais das dependências compiladas para Windows
winutils_url = "https://github.com/cdarlint/winutils/raw/refs/heads/master/hadoop-3.3.6/bin/winutils.exe"
hadoop_dll_url = "https://github.com/cdarlint/winutils/raw/refs/heads/master/hadoop-3.3.6/bin/hadoop.dll"

winutils_path = os.path.join(hadoop_bin, "winutils.exe")
hadoop_dll_path = os.path.join(hadoop_bin, "hadoop.dll")

# Executa o download automático apenas se os arquivos não existirem no disco
if not os.path.exists(winutils_path):
    print("Baixando winutils.exe para o Hadoop local...")
    urllib.request.urlretrieve(winutils_url, winutils_path)

if not os.path.exists(hadoop_dll_path):
    print("Baixando hadoop.dll para o Hadoop local...")
    urllib.request.urlretrieve(hadoop_dll_url, hadoop_dll_path)

# Registra as variáveis de ambiente necessárias para o Spark
os.environ["HADOOP_HOME"] = hadoop_dir
os.environ["PATH"] = hadoop_bin + ";" + os.environ["PATH"]


# ==========================================
# 3. CARREGAMENTO DO AMBIENTE E ARQUIVOS
# ==========================================
# Carrega as variáveis de ambiente do arquivo .env
load_dotenv(dotenv_path="../.env")

True

In [2]:
# Inicializa o Spark com o pacote JDBC do PostgreSQL
spark = (
    SparkSession.builder
    .appName("ML_Ingestion_Profiling")
    .config("spark.jars.packages", "org.postgresql:postgresql:42.7.3")
    .getOrCreate()
)

print("Versão do Spark utilizada:", spark.version)

Versão do Spark utilizada: 4.1.2


In [3]:
db_host = os.getenv("DB_HOST")
db_port = os.getenv("DB_PORT", "5432")
db_name = os.getenv("DB_NAME", "postgres")
db_user = os.getenv("DB_USER")
db_password = os.getenv("DB_PASSWORD")

if not all([db_host, db_user, db_password]):
    raise ValueError("Verifique se as credenciais do Supabase estão configuradas no arquivo .env.")

# URL de conexão JDBC para PostgreSQL
jdbc_url = f"jdbc:postgresql://{db_host}:{db_port}/{db_name}?sslmode=require"

In [4]:
def carregar_tabela_supabase(nome_tabela: str):
    return (
        spark.read
        .format("jdbc")
        .option("url", jdbc_url)
        .option("dbtable", nome_tabela)
        .option("user", db_user)
        .option("password", db_password)
        .option("driver", "org.postgresql.Driver")
        .load()
    )

In [5]:
print("Carregando tabela de contas_gaps...")
df_gaps = carregar_tabela_supabase("public.contas_gaps")

# Armazena a contagem total de linhas para o cálculo de proporção de nulos
total_linhas_gaps = df_gaps.count()
print(f"Total de registros encontrados: {total_linhas_gaps}")

# Calcula a quantidade e o percentual de valores nulos por coluna
null_counts_gaps = df_gaps.select([
    F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df_gaps.columns
]).collect()[0].asDict()

# Ordena o relatório de nulos do maior para o menor percentual
relatorio_nulos_gaps = sorted(
    [(col, qtd, (qtd / total_linhas_gaps) * 100) for col, qtd in null_counts_gaps.items()],
    key=lambda x: x[2],
    reverse=True
)

print("\nRelatório de Valores Nulos (contas_gaps):")
for col, qtd, pct in relatorio_nulos_gaps:
    print(f"Coluna: {col:25} | Nulos: {qtd:8} | Percentual: {pct:6.2f}%")

Carregando tabela de contas_gaps...
Total de registros encontrados: 117016

Relatório de Valores Nulos (contas_gaps):
Coluna: gap_percentual_bucket     | Nulos:    76122 | Percentual:  65.05%
Coluna: id                        | Nulos:        0 | Percentual:   0.00%
Coluna: classe_ativo              | Nulos:        0 | Percentual:   0.00%
Coluna: aderencia_percentual      | Nulos:        0 | Percentual:   0.00%
Coluna: tipo_gap                  | Nulos:        0 | Percentual:   0.00%
Coluna: created_at                | Nulos:        0 | Percentual:   0.00%
Coluna: cod_conta_uuid            | Nulos:        0 | Percentual:   0.00%
Coluna: custodia_faixa            | Nulos:        0 | Percentual:   0.00%


In [6]:
print("Carregando tabela de campanhas_links...")
df_campanhas = carregar_tabela_supabase("public.campanhas_links")

total_linhas_campanhas = df_campanhas.count()
print(f"Total de registros encontrados: {total_linhas_campanhas}")

# Calcula a quantidade e o percentual de valores nulos por coluna
null_counts_campanhas = df_campanhas.select([
    F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df_campanhas.columns
]).collect()[0].asDict()

# Ordena o relatório de nulos do maior para o menor percentual
relatorio_nulos_campanhas = sorted(
    [(col, qtd, (qtd / total_linhas_campanhas) * 100) for col, qtd in null_counts_campanhas.items()],
    key=lambda x: x[2],
    reverse=True
)

print("\nRelatório de Valores Nulos (campanhas_links):")
for col, qtd, pct in relatorio_nulos_campanhas:
    print(f"Coluna: {col:25} | Nulos: {qtd:8} | Percentual: {pct:6.2f}%")

Carregando tabela de campanhas_links...
Total de registros encontrados: 9

Relatório de Valores Nulos (campanhas_links):
Coluna: embedding                 | Nulos:        9 | Percentual: 100.00%
Coluna: id                        | Nulos:        0 | Percentual:   0.00%
Coluna: product_type              | Nulos:        0 | Percentual:   0.00%
Coluna: title                     | Nulos:        0 | Percentual:   0.00%
Coluna: description               | Nulos:        0 | Percentual:   0.00%
Coluna: destaque                  | Nulos:        0 | Percentual:   0.00%
Coluna: created_at                | Nulos:        0 | Percentual:   0.00%
Coluna: embedding_texto           | Nulos:        0 | Percentual:   0.00%


In [7]:
# Estrutura do relatório final consolidado
data_profiling_report = {
    "executado_em": datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC"),
    "contas_gaps": {
        "total_linhas": total_linhas_gaps,
        "detalhe_nulos": {col: {"quantidade": qtd, "percentual": round(pct, 2)} for col, qtd, pct in relatorio_nulos_gaps}
    },
    "campanhas_links": {
        "total_linhas": total_linhas_campanhas,
        "detalhe_nulos": {col: {"quantidade": qtd, "percentual": round(pct, 2)} for col, qtd, pct in relatorio_nulos_campanhas}
    }
}

# Garante que a pasta para salvar os artefatos exista
os.makedirs("../artifacts", exist_ok=True)
caminho_artefato = "../artifacts/profiling_summary.json"

with open(caminho_artefato, "w", encoding="utf-8") as f:
    json.dump(data_profiling_report, f, ensure_ascii=False, indent=2)

print(f"Sumário de Profiling exportado com sucesso em: {caminho_artefato}")

Sumário de Profiling exportado com sucesso em: ../artifacts/profiling_summary.json


C:\Users\mthsn\AppData\Local\Temp\ipykernel_42260\2969711422.py:3: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "executado_em": datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S UTC"),


In [ ]:
spark.stop()
print("Sessão do Spark encerrada com sucesso.")